In [3]:
#%pip install -r requirements-estimation.txt

In [5]:
%pip install -r requirements.txt -q

Note: you may need to restart the kernel to use updated packages.


In [6]:
from eep153_tools.sheets import read_sheets

In [7]:
NG_URL= "https://docs.google.com/spreadsheets/d/1rbWPdn_Rt3MRoFb_wkomHBcNVmM-3F3lNwCpxP5W_3g/edit?gid=1354121303#gid=1354121303" 

In [14]:
import pandas as pd
import numpy as np

FE_Sheets = ["Food Expenditures (2010Q3)", "Food Expenditures (2011Q1)", "Food Expenditures (2012Q3)", "Food Expenditures (2013Q1)", "Food Expenditures (2015Q3)", "Food Expenditures (2016Q1)"]

FP_Sheets = ["Food Prices (2010Q3)", "Food Prices (2011Q1)", "Food Prices (2012Q3)", "Food Prices (2013Q1)", "Food Prices (2015Q3)", "Food Prices (2016Q1)"]

def prepare_data(NG_URL, FE_Sheet, household_sheet="Household Characteristics"): 
    x = read_sheets(NG_URL, sheet = FE_Sheet)
    d = read_sheets(NG_URL ,sheet = household_sheet)

    # data cleaning to ensure key columns match exactly
    for df in [x, d]:
        df['i'] = pd.to_numeric(df['i'], errors='coerce')
        df['t'] = df['t'].astype(str).str.strip()
        df['m'] = (
            df['m']
            .astype(str)
            .str.strip()
            .str.lower()
            .str.replace(r'\s+', ' ', regex=True)
        )


    d.columns.name = 'k'
    # remove duplicate column names, keep first occurrence
    d = d.loc[:, ~d.columns.duplicated()]

    # collapse duplicate household rows
    d = d.groupby(['i','t','m'], as_index=False).first()

    # Data comes in long format; pivot to wide (one column per food)
    x = x.pivot_table(index=['i','t','m'],columns='j',values='Expenditure',
                   aggfunc='sum')
    x = x.replace(0,np.nan) # Replace zeros with missing

    # Take logs of expenditures; call this y
    y = x

    d.set_index(['i','t','m'],inplace=True)

    # NaN in demographic columns means zero people in that category
    d = d.fillna(0)

    return y, d

## [B] Nutritional adequacy of diet
In this section we look at the proportion of household satisfying the adult nutrition requirement in 2010Q3. 

In [18]:
# Extracting 2016 Q1 from y and d
y, d = prepare_data(NG_URL, "Food Expenditures (2010Q3)")
y.head()

,,j,Avocado pear,Baby milk powder,Bananas,Beef,Beer,Bottled water,Bread,Brown beans,Butter/Margarine,Canned beef/corned beef,...,Sugar,Sweet potatoes,Tea,Tomato puree (canned),Tomatoes,Wheat flour,White beans,Wild game meat,Yam flour,Yam--roots
i,t,m,,,,,,,,,,,,,,,,,,,,,
10001,2010Q3,south east,NaN,NaN,133.333333,500.0,160.0,190.0,360.0,600.0,NaN,NaN,...,NaN,NaN,NaN,120.0,100.000000,NaN,300.0,NaN,NaN,800.0
10002,2010Q3,south east,NaN,NaN,116.666667,NaN,1600.0,1140.0,360.0,120.0,NaN,NaN,...,42.857140,NaN,30.0,30.0,80.000000,630.0,210.0,NaN,NaN,400.0
10003,2010Q3,south east,NaN,NaN,100.000000,500.0,NaN,NaN,240.0,NaN,NaN,NaN,...,342.857118,NaN,NaN,180.0,66.666667,NaN,210.0,NaN,NaN,400.0
10004,2010Q3,south east,NaN,NaN,16.666667,500.0,NaN,95.0,240.0,210.0,NaN,NaN,...,42.857140,NaN,NaN,90.0,66.666667,NaN,NaN,NaN,NaN,NaN
10005,2010Q3,south east,NaN,NaN,NaN,1125.0,NaN,NaN,840.0,150.0,200.0,NaN,...,51.428570,NaN,30.0,150.0,NaN,NaN,300.0,NaN,NaN,600.0


In [42]:
d_2010Q3 = d.xs('2010Q3', level='t').drop(columns = ['Female NA', 'Male NA'])
d_2010Q3.head()

,k,Female 00-03,Female 04-08,Female 09-13,Female 14-18,Female 19-30,Female 31-50,Female 51+,Male 00-03,Male 04-08,Male 09-13,Male 14-18,Male 19-30,Male 31-50,Male 51+,log HSize
i,m,,,,,,,,,,,,,,,
10001,south east,1,0,0,0,1,2,0,0,0,0,0,1,2,0,1.945910
10002,south east,0,0,0,0,2,1,0,0,0,1,1,1,1,0,1.945910
10003,south east,0,0,1,0,0,1,0,0,1,1,1,0,0,1,1.791759
10004,south east,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0.693147
10005,south east,0,2,0,0,1,1,0,0,0,1,0,0,1,0,1.791759


In [39]:
#Minimum Dietary Nutrition Requirement

RDIs = pd.read_csv('https://docs.google.com/spreadsheets/d/e/2PACX-1vR84qu7yUvdzhiHnPG_7KpVSx9FgiyLrZqJ8Ft3oK-B6xT6i4xN8CNtZ2c9PRGXqxXQCNorzWdF6NnB/pub?gid=0&single=true&output=csv')
bmin = RDIs.set_index('Nutrition')
bmin.head()

,Female 00-03,Female 04-08,Female 09-13,Female 14-18,Female 19-30,Female 31-50,Female 51+,Male 00-03,Male 04-08,Male 09-13,Male 14-18,Male 19-30,Male 31-50,Male 51+
Nutrition,,,,,,,,,,,,,,
Energy,1000.0,1200.0,1600.0,1800.0,2000.0,1800.0,1600.0,1000.0,1400.0,1800.0,2200.0,2400.0,2200.0,2000.0
Protein,13.0,19.0,34.0,46.0,46.0,46.0,46.0,13.0,19.0,34.0,52.0,56.0,56.0,56.0
"Fiber, total dietary",14.0,16.8,22.4,25.2,28.0,25.2,22.4,14.0,19.6,25.2,30.8,33.6,30.8,28.0
"Folate, DFE",150.0,200.0,300.0,400.0,400.0,400.0,400.0,150.0,200.0,300.0,400.0,400.0,400.0,400.0
"Calcium, Ca",700.0,1000.0,1300.0,1300.0,1000.0,1000.0,1200.0,700.0,1000.0,1300.0,1300.0,1000.0,1000.0,1000.0


In [55]:
frames = {}
for nutrient in bmin.index:
    # Multiply the whole household df by the specific row of requirements
    frames[nutrient] = d_2010Q3.multiply(bmin.loc[nutrient], axis=1)
household_rdi = pd.concat(frames, names=['Nutrient']).reorder_levels([1, 2, 0]).sort_index()


In [56]:
household_rdi.head()

k                                             Female 00-03  Female 04-08  \
i     m          Nutrient                                                  
10001 south east Calcium, Ca                         700.0           0.0   
                 Carbohydrate, by difference         130.0           0.0   
                 Energy                             1000.0           0.0   
                 Fiber, total dietary                 14.0           0.0   
                 Folate, DFE                         150.0           0.0   

k                                             Female 09-13  Female 14-18  \
i     m          Nutrient                                                  
10001 south east Calcium, Ca                           0.0           0.0   
                 Carbohydrate, by difference           0.0           0.0   
                 Energy                                0.0           0.0   
                 Fiber, total dietary                  0.0           0.0   
                 Folate, DFE                           0.0           0.0   

k                                             Female 19-30  Female 31-50  \
i     m          Nutrient                                                  
10001 south east Calcium, Ca                        1000.0        2000.0   
                 Carbohydrate, by difference         130.0         260.0   
                 Energy                             2000.0        3600.0   
                 Fiber, total dietary                 28.0          50.4   
                 Folate, DFE                         400.0         800.0   

k                                             Female 51+  Male 00-03  \
i     m          Nutrient                                              
10001 south east Calcium, Ca                         0.0         0.0   
                 Carbohydrate, by difference         0.0         0.0   
                 Energy                              0.0         0.0   
                 Fiber, total dietary                0.0         0.0   
                 Folate, DFE                         0.0         0.0   

k                                             Male 04-08  Male 09-13  \
i     m          Nutrient                                              
10001 south east Calcium, Ca                         0.0         0.0   
                 Carbohydrate, by difference         0.0         0.0   
                 Energy                              0.0         0.0   
                 Fiber, total dietary                0.0         0.0   
                 Folate, DFE                         0.0         0.0   

k                                             Male 14-18  Male 19-30  \
i     m          Nutrient                                              
10001 south east Calcium, Ca                         0.0      1000.0   
                 Carbohydrate, by difference         0.0       130.0   
                 Energy                              0.0      2400.0   
                 Fiber, total dietary                0.0        33.6   
                 Folate, DFE                         0.0       400.0   

k                                             Male 31-50  Male 51+  log HSize  
i     m          Nutrient                                                      
10001 south east Calcium, Ca                      2000.0       0.0        NaN  
                 Carbohydrate, by difference       260.0       0.0        NaN  
                 Energy                           4400.0       0.0        NaN  
                 Fiber, total dietary               61.6       0.0        NaN  
                 Folate, DFE                       800.0       0.0        NaN